# Liu2024 Cohort Class-Feature Audit

Configurable audit of physiological, fixed Riemannian, frozen S-JEPA, and explicitly all-cohort transductive microstate separation. The default cohort is the mapped Lv14; `subjects_to_use=None` selects all 50 discovered Liu2024 subjects.


# 1. Setup

In [ ]:
import os, sys, json, random, hashlib, builtins, platform, inspect, itertools
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import signal, stats
from scipy.optimize import linear_sum_assignment
from sklearn.covariance import OAS
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
print(f'Python: {sys.version.split()[0]} | Platform: {platform.platform()} | cwd: {Path.cwd()}')

# 2. Configuration
## 2.1 Domain Defaults
## 2.2 CONFIG

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent
CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    'artifact_dir': str(WORKING_DIR / 'artifacts' / 'liu2024-lv14-class-feature-audit'),
    'source_extract_dir': str(WORKING_DIR / 'liu2024_data' / 'liu2024_figshare' / 'sourcedata'),
    'experiment_name': 'lv14_class_feature_audit_all_branches',
    'cohort_name': 'lv14',
    'config_note': 'Mapped Lv14 class-feature audit; microstate maps are cohort-transductive.',
    'locked_shallow_subject_results': str(WORKING_DIR / 'artifacts' / 'liu2024-compact-mi-models' / '20260712_165746_790145_fd8ab986' / 'subject_results.csv'),
    'locked_prelocal_subject_metrics': str(WORKING_DIR / 'artifacts' / 'liu2024-sjepa-prelocal' / '20260703_0835_5d8bfb69' / 'subject_metrics.json'),
    # ------------------------------------------------------------------
    # Dataset and independent full-trial preprocessing
    # ------------------------------------------------------------------
    'subjects_to_use': [1,3,7,9,10,11,14,15,17,29,31,32,37,41],
    'flagged_subjects': [15], 'sfreq': 500, 'marker_channel_index': 32, 'marker_value': 2,
    'onset_plausible_range': [800,1300], 'onset_fallback_sample': 1003, 'mi_window_s': [0.0,4.0],
    'filter_order': 4, 'average_reference': True, 'bands_hz': [[8,12],[13,20],[20,30],[8,30]],
    # ------------------------------------------------------------------
    # Evaluation and inference
    # ------------------------------------------------------------------
    'cv_folds': 5, 'cv_random_state': 2026, 'lda_shrinkage': 'auto', 'permutation_iterations': 2000,
    'bootstrap_iterations': 5000, 'bootstrap_seed': 202607, 'split_half_repeats': 100,
    'split_half_seed': 202608, 'correlation_permutations': 20000,
    # ------------------------------------------------------------------
    # Frozen S-JEPA
    # ------------------------------------------------------------------
    'enable_sjepa': True, 'sjepa_model_id': 'braindecode/signal-jepa_without-chans',
    'sjepa_model_revision': None, 'sjepa_checkpoint_path': None, 'sjepa_sfreq': 128,
    'sjepa_bandpass_hz': [0.5,40.0], 'sjepa_hook': 'feature_encoder', 'sjepa_pooling': 'mean',
    'sjepa_embedding_dim': 64, 'sjepa_batch_size': 16, 'device': 'auto',
    # ------------------------------------------------------------------
    # Transductive microstates
    # ------------------------------------------------------------------
    'state_naming_mode': 'neutral', 'microstate_sfreq': 250, 'microstate_band_hz': [1.0,30.0],
    'n_states': 4, 'gfp_min_peak_distance_ms': 30.0, 'gfp_max_peaks': 20000, 'kmeans_n_init': 20,
    'allow_transductive_group_maps': True,
    # ------------------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------------------
    'seed': 2026, 'set_seed': True,
}
LV14_DEFAULT = [1,3,7,9,10,11,14,15,17,29,31,32,37,41]
LIU29 = ['Fp1','Fp2','Fz','F3','F4','F7','F8','FCz','FC3','FC4','FT7','FT8','Cz','C3','C4','T3','T4','CP3','CP4','TP7','TP8','Pz','P3','P4','T5','T6','Oz','O1','O2']
LIU29_IDX = [i for i in range(30) if i != 17]
MOTOR13 = ['F3','F4','FCz','FC3','FC4','Cz','C3','C4','CP3','CP4','Pz','P3','P4']
PAIRS = [('C3','C4'),('FC3','FC4'),('CP3','CP4'),('P3','P4')]
assert len(LIU29) == len(LIU29_IDX) == 29
print(f"Configured cohort: {CONFIG['cohort_name']} | requested subjects: {CONFIG['subjects_to_use']} | S-JEPA: {CONFIG['enable_sjepa']} | state naming: {CONFIG['state_naming_mode']}")

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f'{timestamp}_{config_hash}'
RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG['artifact_dir']) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = ARTIFACT_DIR / 'run.log'
_LOG_FILE_HANDLE = open(LOG_PATH, 'a', buffering=1, encoding='utf-8', errors='replace')
def _safe_write_text(stream, text):
    try: stream.write(text)
    except UnicodeEncodeError:
        encoding = getattr(stream, 'encoding', None) or 'utf-8'
        stream.write(text.encode(encoding, errors='replace').decode(encoding, errors='replace'))
def _timestamped_print(*args, **kwargs):
    sep, end = kwargs.pop('sep', ' '), kwargs.pop('end', '\n')
    flush, target = kwargs.pop('flush', False), kwargs.pop('file', None)
    message = sep.join(str(a) for a in args); stamped = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}" if message else ''
    for stream in ([target] if target is not None else [sys.stdout, _LOG_FILE_HANDLE]): _safe_write_text(stream, stamped + end)
    if flush: _LOG_FILE_HANDLE.flush()
builtins.print = _timestamped_print
config_path = ARTIFACT_DIR / 'config.json'
with open(config_path, 'w') as f: json.dump(CONFIG, f, indent=2, allow_nan=False)
print(f'Run ID:     {RUN_ID}')
print(f'Artifacts:  {ARTIFACT_DIR}')
print(f'Config:     {config_path}')

## 2.4 Reproducibility

In [ ]:
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed); random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        torch.use_deterministic_algorithms(True, warn_only=True)
    except ImportError: pass
BASE_SEED = int(CONFIG['seed'])
if CONFIG['set_seed']: seed_everything(BASE_SEED)
print(f'Seed initialized: {BASE_SEED}')

# 3. Load and Prepare Data
## 3.1 Data Loading, Statistics, and Synthetic Checks

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def selected_files():
    files=sorted(Path(CONFIG['source_extract_dir']).glob('sub-*/sub-*_eeg.mat'))
    discovered={}
    for p in files:
        sid=int(p.parent.name.split('-')[1])
        if sid in discovered: raise FileExistsError(f'Multiple source files discovered for sub-{sid:02d}: {discovered[sid]} and {p}')
        discovered[sid]=p
    requested=CONFIG.get('subjects_to_use')
    expected=sorted(discovered) if requested is None else sorted(int(s) for s in requested)
    if requested is None and expected != list(range(1,51)):
        raise FileNotFoundError(f'subjects_to_use=None requires exactly discovered sub-01..sub-50; found {expected}')
    missing=[sid for sid in expected if sid not in discovered]
    if missing: raise FileNotFoundError(f'Configured subjects missing source files: {missing}')
    out=[discovered[sid] for sid in expected]
    found=[int(p.parent.name.split('-')[1]) for p in out]
    if found != expected: raise AssertionError(f'Selected file subjects {found} do not match configured subjects {expected}')
    return out
def load_subject(path):
    eeg=sio.loadmat(path)['eeg'][0,0]; raw=np.asarray(eeg['rawdata'],float); y=np.asarray(eeg['label']).ravel().astype(int)-1
    if raw.ndim != 3 or raw.shape[1] < 33 or set(np.unique(y)) != {0,1}: raise ValueError(f'Unexpected Liu data in {path}: {raw.shape}, labels={np.unique(y)}')
    marker=raw[:,CONFIG['marker_channel_index'],:]; lo,hi=CONFIG['onset_plausible_range']; detected=[]
    for m in marker:
        hits=np.flatnonzero(m==CONFIG['marker_value']); valid=hits[(hits>=lo)&(hits<=hi)]; detected.append(int(valid[0]) if len(valid) else -1)
    plausible=[v for v in detected if lo<=v<=hi]; fallback=int(np.median(plausible)) if plausible else CONFIG['onset_fallback_sample']
    onsets=np.asarray([v if lo<=v<=hi else fallback for v in detected]); sid=int(path.parent.name.split('-')[1])
    ids=np.asarray([f'sub-{sid:02d}_trial-{i:03d}' for i in range(len(y))])
    return {'subject_id':sid,'raw':raw[:,LIU29_IDX,:].copy(),'labels':y,'onsets':onsets,'trial_ids':ids,'path':path,'sha256':sha256_file(path)}
def bh(p):
    p=np.asarray(p,float); order=np.argsort(p); ranked=p[order]; qrank=np.minimum.accumulate((ranked*len(p)/np.arange(1,len(p)+1))[::-1])[::-1]; q=np.empty_like(qrank); q[order]=np.minimum(qrank,1); return q
def hedges_g(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float); pooled=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2)); d=(b.mean()-a.mean())/(pooled+1e-12); return d*(1-3/(4*(len(a)+len(b))-9))
def rank_biserial(a,b):
    u=stats.mannwhitneyu(b,a,alternative='two-sided').statistic; return 2*u/(len(a)*len(b))-1
def bootstrap_effect_ci(a,b,fn,rng,n=None):
    n=n or CONFIG['bootstrap_iterations']; vals=[fn(rng.choice(a,len(a),True),rng.choice(b,len(b),True)) for _ in range(n)]; return np.quantile(vals,[.025,.975])
def permutation_p(a,b,fn,rng,n=None):
    n=n or CONFIG['permutation_iterations']; z=np.r_[a,b]; observed=abs(fn(a,b)); count=0
    for _ in range(n):
        q=rng.permutation(z); count += abs(fn(q[:len(a)],q[len(a):])) >= observed-1e-15
    return (count+1)/(n+1)
def permutation_corr(x,y,method,n_perm,seed):
    fn=stats.spearmanr if method=='spearman' else stats.kendalltau; obs=float(fn(x,y).statistic); centered=np.asarray(y)-np.mean(y); extreme=0
    if len(x)<=20:
        signs_iter=itertools.product([-1.,1.],repeat=len(x)); total=2**len(x); method_name='exhaustive centered-performance sign flips'
    else:
        rng=np.random.default_rng(seed); total=int(n_perm); signs_iter=(rng.choice([-1.,1.],len(x)) for _ in range(total)); method_name=f'Monte Carlo centered-performance sign flips ({total} iterations)'
    for signs in signs_iter:
        yp=np.mean(y)+centered*np.asarray(signs); val=float(fn(x,yp).statistic); extreme += abs(val) >= abs(obs)-1e-15
    p=extreme/total if len(x)<=20 else (extreme+1)/(total+1)
    return obs,p,method_name
def synthetic_checks():
    rng=np.random.default_rng(9); a=rng.normal(0,1,20); b=rng.normal(1,1,20)
    assert hedges_g(a,b)>0 and rank_biserial(a,b)>0 and permutation_p(a,b,lambda x,y:np.mean(y)-np.mean(x),rng,199)<.1
    assert np.all(bh([.001,.02,.8])>=0) and np.all(bh([.001,.02,.8])<=1)
synthetic_checks(); print('Synthetic feature/effect/permutation checks passed.')

## 3.2 Independent Full-Trial Filtering and Physiological Features

In [ ]:
def filtered_mi(raw,onsets,band,sfreq=None):
    sfreq=sfreq or CONFIG['sfreq']; sos=signal.butter(CONFIG['filter_order'],band,btype='bandpass',fs=sfreq,output='sos'); out=[]; n=int(4*sfreq)
    for trial,onset in zip(raw,onsets):
        x=trial-trial.mean(0,keepdims=True) if CONFIG['average_reference'] else trial.copy(); z=signal.sosfiltfilt(sos,x,axis=-1); crop=z[:,onset:onset+n]
        if crop.shape[-1]!=n: raise ValueError('Incomplete marker-relative 0-4 s crop after full-trial filtering')
        out.append(crop)
    return np.asarray(out)
def physiology_features(d):
    rows=[]; name_idx={n:i for i,n in enumerate(LIU29)}
    for lo,hi in CONFIG['bands_hz']:
        x=filtered_mi(d['raw'],d['onsets'],[lo,hi]); power=np.log(np.mean(x*x,axis=-1)+1e-20); band=f'{lo}-{hi}Hz'
        for i in range(len(x)):
            row={'subject_id':d['subject_id'],'trial_id':d['trial_ids'][i],'label':int(d['labels'][i]),'hand':'right' if d['labels'][i] else 'left','band':band}
            for ch in LIU29: row[f'logbp_{ch}']=power[i,name_idx[ch]]
            for a,b in PAIRS: row[f'asym_{a}_{b}']=power[i,name_idx[a]]-power[i,name_idx[b]]
            left=[name_idx[n] for n in ['F3','FC3','C3','CP3','P3']]; right=[name_idx[n] for n in ['F4','FC4','C4','CP4','P4']]; mid=[name_idx[n] for n in ['FCz','Cz','Pz']]
            row.update({'motor13_left':power[i,left].mean(),'motor13_right':power[i,right].mean(),'motor13_midline':power[i,mid].mean(),'motor13_global':power[i,[name_idx[n] for n in MOTOR13]].mean()}); rows.append(row)
    return rows
def physiology_effects(df):
    rows=[]; rng=np.random.default_rng(CONFIG['bootstrap_seed']); keys=[c for c in df if c.startswith(('logbp_','asym_','motor13_'))]
    for (sid,band),g in df.groupby(['subject_id','band']):
        for feature in keys:
            a=g.loc[g.label==0,feature].to_numpy(); b=g.loc[g.label==1,feature].to_numpy(); family='channel_power' if feature.startswith('logbp') else ('asymmetry' if feature.startswith('asym') else 'motor13_summary')
            lo,hi=bootstrap_effect_ci(a,b,hedges_g,rng); rows.append({'subject_id':sid,'band':band,'feature':feature,'family':family,'mean_left':a.mean(),'mean_right':b.mean(),'difference_right_minus_left':b.mean()-a.mean(),'hedges_g':hedges_g(a,b),'cliff_rank_biserial':rank_biserial(a,b),'bootstrap_ci_low':lo,'bootstrap_ci_high':hi,'permutation_p':permutation_p(a,b,lambda x,y:np.mean(y)-np.mean(x),rng)})
        for a,b in PAIRS:
            f=f'asym_{a}_{b}'; left=g.loc[g.label==0,f].to_numpy(); right=g.loc[g.label==1,f].to_numpy(); diff=right.mean()-left.mean(); ilo,ihi=bootstrap_effect_ci(left,right,hedges_g,rng); rows.append({'subject_id':sid,'band':band,'feature':f'hand_by_hemisphere_{a}_{b}','family':'hand_by_hemisphere','mean_left':left.mean(),'mean_right':right.mean(),'difference_right_minus_left':diff,'hedges_g':hedges_g(left,right),'cliff_rank_biserial':rank_biserial(left,right),'bootstrap_ci_low':ilo,'bootstrap_ci_high':ihi,'permutation_p':permutation_p(left,right,lambda x,y:np.mean(y)-np.mean(x),rng)})
    out=pd.DataFrame(rows); out['bh_q']=out.groupby(['subject_id','band','family'])['permutation_p'].transform(lambda p:bh(p)); out['correction']='BH within subject x band x declared family'; return out

def channel_physiology_effects(effects):
    out=effects[effects.feature.isin([f'logbp_{ch}' for ch in LIU29])].copy()
    out['channel']=out.feature.str.removeprefix('logbp_')
    return out[['subject_id','band','channel','mean_left','mean_right','difference_right_minus_left','hedges_g','cliff_rank_biserial','bootstrap_ci_low','bootstrap_ci_high','permutation_p','bh_q','correction']]

def physiology_cohort_effects(effects):
    prespecified=[f'logbp_{ch}' for ch in LIU29]+[f'asym_{a}_{b}' for a,b in PAIRS]+[f'hand_by_hemisphere_{a}_{b}' for a,b in PAIRS]
    rows=[]; rng=np.random.default_rng(CONFIG['bootstrap_seed']+19)
    for (band,feature),g in effects[effects.feature.isin(prespecified)].groupby(['band','feature'],sort=False):
        values=g.set_index('subject_id').reindex(SUBJECTS).difference_right_minus_left.to_numpy(float)
        if np.isnan(values).any(): raise AssertionError(f'Missing configured-subject effects for {band} {feature}')
        boots=np.asarray([np.mean(rng.choice(values,len(values),True)) for _ in range(CONFIG['bootstrap_iterations'])])
        w=stats.wilcoxon(values) if np.any(values) else type('R',(),{'statistic':0.,'pvalue':1.})()
        rows.append({'band':band,'feature':feature,'family':'channel_x_band_and_prespecified_homologous_asymmetry','n_subjects':len(values),'mean_difference_right_minus_left':values.mean(),'mean_subject_hedges_g':g.hedges_g.mean(),'bootstrap_ci_low':np.quantile(boots,.025),'bootstrap_ci_high':np.quantile(boots,.975),'wilcoxon_stat':w.statistic,'wilcoxon_p':w.pvalue})
    out=pd.DataFrame(rows); out['bh_q']=bh(out.wilcoxon_p); out['correction']='BH across all channel x band and prespecified homologous/asymmetry cohort tests'; return out



# 4. Model
## 4.1 Fixed Motor13 Riemann and Frozen S-JEPA Features

In [ ]:
def motor13_covariances(d):
    idx=[LIU29.index(n) for n in MOTOR13]; x=filtered_mi(d['raw'][:,idx],d['onsets'],[8,30]); specs={'4s':[(0,4)],'2s':[(0,2),(2,4)],'1s':[(0,1),(1,2),(2,3),(3,4)]}; views={}
    for name,windows in specs.items():
        arr=[]
        for trial in x:
            cs=[]
            for a,b in windows: cs.append(OAS(store_precision=False,assume_centered=True).fit(trial[:,int(a*CONFIG['sfreq']):int(b*CONFIG['sfreq'])].T).covariance_)
            arr.append(np.mean(cs,axis=0))
        views[name]=np.asarray(arr)
    return views
def fit_riemann_fold(views,y,tr,te):
    from pyriemann.tangentspace import TangentSpace
    ztr=[]; zte=[]
    for name in ['4s','2s','1s']:
        ts=TangentSpace(metric='riemann'); ztr.append(ts.fit_transform(views[name][tr])); zte.append(ts.transform(views[name][te]))
    a,b=np.c_[*ztr],np.c_[*zte]; scaler=StandardScaler().fit(a); a=scaler.transform(a); b=scaler.transform(b); clf=LinearDiscriminantAnalysis(solver='lsqr',shrinkage=CONFIG['lda_shrinkage']).fit(a,y[tr]); score=clf.decision_function(b); pred=(score>=0).astype(int); centroids=np.stack([a[y[tr]==k].mean(0) for k in [0,1]]); dist=np.linalg.norm(b[:,None,:]-centroids[None,:,:],axis=2); return pred,score,dist
def resolve_sjepa_revision():
    if CONFIG['sjepa_checkpoint_path']: return CONFIG['sjepa_model_revision']
    if CONFIG['sjepa_model_revision']: return CONFIG['sjepa_model_revision']
    try:
        from huggingface_hub import scan_cache_dir
        repos=[r for r in scan_cache_dir().repos if r.repo_id==CONFIG['sjepa_model_id']]; commits=sorted({r.commit_hash for repo in repos for r in repo.revisions})
    except Exception as e: raise RuntimeError('S-JEPA requested but cached revision could not be inspected; set sjepa_model_revision explicitly') from e
    if len(commits)!=1: raise RuntimeError(f'S-JEPA strict provenance requires exactly one cached revision or explicit revision; found {commits}')
    return commits[0]
def sjepa_preprocess(d):
    x=filtered_mi(d['raw'],d['onsets'],CONFIG['sjepa_bandpass_hz']); return signal.resample_poly(x,CONFIG['sjepa_sfreq'],CONFIG['sfreq'],axis=-1).astype('float32')
def extract_sjepa(all_data):
    import torch
    from braindecode.models import SignalJEPA_PreLocal
    revision=resolve_sjepa_revision(); X=np.concatenate([sjepa_preprocess(d) for d in all_data]); labels=np.concatenate([d['labels'] for d in all_data]); ids=np.concatenate([d['trial_ids'] for d in all_data]); subjects=np.concatenate([[d['subject_id']]*len(d['labels']) for d in all_data]); source=[{'subject_id':d['subject_id'],'path':str(d['path']),'sha256':d['sha256']} for d in all_data]
    kwargs={'n_chans':29,'chs_info':None,'n_times':X.shape[-1],'n_outputs':2}; checkpoint=CONFIG['sjepa_checkpoint_path']
    if checkpoint:
        model=SignalJEPA_PreLocal(**kwargs); state=torch.load(checkpoint,map_location='cpu',weights_only=False); state=state.get('state_dict',state) if isinstance(state,dict) else state; model.load_state_dict(state,strict=False)
    else: model=SignalJEPA_PreLocal.from_pretrained(CONFIG['sjepa_model_id'],revision=revision,strict=False,**kwargs)
    for p in model.parameters(): p.requires_grad=False
    device=('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')) if CONFIG['device']=='auto' else CONFIG['device']; model=model.to(device).eval(); captured={}
    hook=getattr(model,CONFIG['sjepa_hook']).register_forward_hook(lambda m,i,o:captured.__setitem__('z',o.detach())); features=[]
    try:
        with torch.no_grad():
            for start in range(0,len(X),CONFIG['sjepa_batch_size']):
                logits=model(torch.from_numpy(X[start:start+CONFIG['sjepa_batch_size']]).to(device)); z=captured.get('z')
                if z is None: raise RuntimeError('feature_encoder hook produced no value')
                if z.ndim>2: z=z.mean(dim=1)
                z=z.flatten(1)
                if z.shape[1]!=CONFIG['sjepa_embedding_dim'] or z.shape==logits.shape or z.shape[1]==2: raise RuntimeError(f'Expected non-logit 64D feature_encoder embeddings, got z={tuple(z.shape)}, logits={tuple(logits.shape)}')
                features.append(z.cpu().numpy())
    finally: hook.remove()
    manifest={'model_id':CONFIG['sjepa_model_id'],'requested_revision':CONFIG['sjepa_model_revision'],'resolved_revision':revision,'checkpoint_path':checkpoint,'checkpoint_sha256':sha256_file(checkpoint) if checkpoint else None,'hook':'feature_encoder','pooling':'mean','embedding_dim':64,'channels':LIU29,'preprocessing':{'independent_full_trial_filter_hz':CONFIG['sjepa_bandpass_hz'],'marker_relative_window_s':[0,4],'source_sfreq':500,'target_sfreq':128,'average_reference':CONFIG['average_reference']},'labels':labels.tolist(),'trial_ids':ids.tolist(),'subjects':subjects.tolist(),'sources':source}
    features=np.concatenate(features).astype('float32'); np.savez_compressed(ARTIFACT_DIR/'sjepa_embeddings.npz',features=features,labels=labels,trial_ids=ids,subjects=subjects,manifest_json=np.asarray(json.dumps(manifest,sort_keys=True))); return features,labels,ids,subjects,manifest
def fit_vector_oof(X,y,sid,branch):
    rows=[]; splitter=StratifiedKFold(CONFIG['cv_folds'],shuffle=True,random_state=CONFIG['cv_random_state']); seen=np.zeros(len(y),int)
    for fold,(tr,te) in enumerate(splitter.split(X,y)):
        scaler=StandardScaler().fit(X[tr]); a,b=scaler.transform(X[tr]),scaler.transform(X[te]); clf=LinearDiscriminantAnalysis(solver='lsqr',shrinkage=CONFIG['lda_shrinkage']).fit(a,y[tr]); score=clf.decision_function(b); pred=(score>=0).astype(int); seen[te]+=1
        for j,k in enumerate(te): rows.append({'subject_id':sid,'trial_index':int(k),'fold_id':fold,'label':int(y[k]),'prediction':int(pred[j]),'score':float(score[j]),'branch':branch})
    if not np.all(seen==1): raise AssertionError(f'{branch} OOF coverage is not exactly once for sub-{sid:02d}: {seen.tolist()}')
    return rows

## 4.2 Four-State Transductive Microstate Features
Group maps use every subject in the configured cohort. Results are descriptive and all-cohort transductive, never inductive classifier evidence. Neutral states make no canonical A/B/C/D claim.


In [ ]:
def micro_preprocess(d):
    x=filtered_mi(d['raw'],d['onsets'],CONFIG['microstate_band_hz']); return signal.resample_poly(x,CONFIG['microstate_sfreq'],CONFIG['sfreq'],axis=-1)*1e-6
def geometry_seeds():
    pos={'Fp1':(-1,3),'Fp2':(1,3),'Fz':(0,2),'F3':(-1,2),'F4':(1,2),'F7':(-2,2),'F8':(2,2),'FCz':(0,1),'FC3':(-1,1),'FC4':(1,1),'FT7':(-2,1),'FT8':(2,1),'Cz':(0,0),'C3':(-1,0),'C4':(1,0),'T3':(-2,0),'T4':(2,0),'CP3':(-1,-1),'CP4':(1,-1),'TP7':(-2,-1),'TP8':(2,-1),'Pz':(0,-2),'P3':(-1,-2),'P4':(1,-2),'T5':(-2,-2),'T6':(2,-2),'Oz':(0,-3),'O1':(-1,-3),'O2':(1,-3)}; xy=np.asarray([pos[n] for n in LIU29],float); x,y=xy[:,0],xy[:,1]; seeds=np.stack([x+y,x-y,y,x*y]); seeds-=seeds.mean(1,keepdims=True); seeds/=np.linalg.norm(seeds,axis=1,keepdims=True); assert np.max(np.abs(seeds@seeds.T-np.eye(4)))<0.99, 'Proxy seeds must remain distinct under polarity invariance'; return seeds
def micro_features_trials(x,centers,names):
    durations={n:[] for n in names}; occurrences={n:0 for n in names}; samples={n:0 for n in names}; transitions={(a,b):0 for a in names for b in names if a!=b}; origins={n:0 for n in names}; total=0; total_s=len(x)*x.shape[-1]/CONFIG['microstate_sfreq']
    for trial in x:
        z=trial-trial.mean(0,keepdims=True); den=(np.linalg.norm(centers,axis=1)[:,None]+1e-12)*(np.linalg.norm(z,axis=0)[None,:]+1e-12); labels=np.abs(centers@z/den).argmax(0); total+=len(labels)
        for k,n in enumerate(names):
            mask=labels==k; starts=np.flatnonzero(mask & np.r_[True,~mask[:-1]]); ends=np.flatnonzero(mask & np.r_[~mask[1:],True]); durations[n].extend(((ends-starts+1)/CONFIG['microstate_sfreq']).tolist()); occurrences[n]+=len(starts); samples[n]+=mask.sum()
        for a,b in zip(labels[:-1],labels[1:]):
            if a!=b: transitions[(names[a],names[b])]+=1; origins[names[a]]+=1
    f={}
    for n in names: f[f'{n}_duration']=np.mean(durations[n]) if durations[n] else 0.; f[f'{n}_occurrence']=occurrences[n]/total_s; f[f'{n}_coverage']=samples[n]/total
    for a in names:
        for b in names:
            if a!=b: f[f'{a}_to_{b}']=transitions[(a,b)]/max(1,origins[a])
    assert len(f)==24 and not any(f'{n}_to_{n}' in f for n in names); return f
def build_microstates(all_data):
    if not CONFIG['allow_transductive_group_maps']: raise RuntimeError('Microstate group maps are all-cohort transductive; explicit opt-in required')
    try:
        import mne
        from pycrostates.cluster import ModKMeans
        from pycrostates.preprocessing import extract_gfp_peaks
        from pycrostates.io import ChData
    except ImportError as e: raise ImportError('Microstate branch requires mne and pycrostates') from e
    xs={}; templates=[]; info=mne.create_info(LIU29,CONFIG['microstate_sfreq'],['eeg']*29)
    for d in all_data:
        x=micro_preprocess(d); xs[d['subject_id']]=x; raw=mne.io.RawArray(x.transpose(1,0,2).reshape(29,-1),info,verbose=False); kwargs={'picks':'eeg','min_peak_distance':max(1,round(CONFIG['gfp_min_peak_distance_ms']*CONFIG['microstate_sfreq']/1000))}; peaks=extract_gfp_peaks(raw,**kwargs); km=ModKMeans(4,n_init=CONFIG['kmeans_n_init'],random_state=BASE_SEED); km.fit(peaks,verbose=False); templates.extend(km.cluster_centers_)
    meta=ModKMeans(4,n_init=CONFIG['kmeans_n_init'],random_state=BASE_SEED); meta.fit(ChData(np.asarray(templates).T,info),verbose=False); centers=np.asarray(meta.cluster_centers_); neutral=[f'MS{i}' for i in range(4)]; names=neutral; rows=[]
    if CONFIG['state_naming_mode']=='proxy_canonical':
        corr=np.abs(np.corrcoef(centers,geometry_seeds())[:4,4:]); rr,cc=linear_sum_assignment(-corr); mapping={neutral[r]:'ABCD'[c] for r,c in zip(rr,cc)}; names=[mapping[n] for n in neutral]
        for i,n in enumerate(neutral):
            row={'state':n,'assigned_proxy':mapping[n],'status':'proxy_canonical_not_exact',**{f'corr_{c}':corr[i,j] for j,c in enumerate('ABCD')}}; row.update({f'center_{ch}':centers[i,j] for j,ch in enumerate(LIU29)}); rows.append(row)
    elif CONFIG['state_naming_mode']=='neutral':
        for i,n in enumerate(neutral):
            row={'state':n,'assigned_proxy':None,'status':'neutral_not_matched',**{f'corr_{c}':np.nan for c in 'ABCD'}}; row.update({f'center_{ch}':centers[i,j] for j,ch in enumerate(LIU29)}); rows.append(row)
    else: raise ValueError('state_naming_mode must be neutral or proxy_canonical')
    matching=pd.DataFrame(rows); feats=[]
    for d in all_data:
        for label in [0,1]: feats.append({'subject_id':d['subject_id'],'label':label,'hand':'right' if label else 'left','transductive_group_maps':True,'state_naming_mode':CONFIG['state_naming_mode'],**micro_features_trials(xs[d['subject_id']][d['labels']==label],centers,names)})
    long=pd.DataFrame(feats).melt(id_vars=['subject_id','label','hand','transductive_group_maps','state_naming_mode'],var_name='feature',value_name='value'); assert long.feature.nunique()==24
    return long,matching

# 5. Training
## 5.1 Exactly-Once OOF and Per-Subject Effects
## 5.2 Run All Subjects

In [ ]:
print('Full config:\n'+json.dumps(CONFIG,indent=2,sort_keys=True))
SELECTED_FILES=selected_files(); ALL_DATA=[load_subject(p) for p in SELECTED_FILES]; SUBJECTS=[d['subject_id'] for d in ALL_DATA]
CONFIG_SUBJECTS=sorted(int(s) for s in CONFIG['subjects_to_use']) if CONFIG.get('subjects_to_use') is not None else list(range(1,51))
assert SUBJECTS==CONFIG_SUBJECTS, f'Loaded subjects {SUBJECTS} != configured subjects {CONFIG_SUBJECTS}'
if CONFIG.get('subjects_to_use') is None: assert SUBJECTS==list(range(1,51)) and len(SUBJECTS)==50
inventory=pd.DataFrame([{'subject_id':d['subject_id'],'n_trials':len(d['labels']),'n_left':int((d['labels']==0).sum()),'n_right':int((d['labels']==1).sum()),'flagged_sub15_sensitivity':d['subject_id']==15,'source_path':str(d['path']),'source_sha256':d['sha256']} for d in ALL_DATA]); inventory.to_csv(ARTIFACT_DIR/'subject_inventory.csv',index=False)
PHYS=pd.DataFrame(sum((physiology_features(d) for d in ALL_DATA),[])); PHYS_EFFECTS=physiology_effects(PHYS)
PHYS_CHANNEL_EFFECTS=channel_physiology_effects(PHYS_EFFECTS); PHYS_COHORT_EFFECTS=physiology_cohort_effects(PHYS_EFFECTS)
RIEMANN_ROWS=[]; RIEMANN_SEP=[]; RIEMANN_VIEWS={}
for d in ALL_DATA:
    sid=d['subject_id']; y=d['labels']; views=motor13_covariances(d); RIEMANN_VIEWS[sid]=views; splitter=StratifiedKFold(5,shuffle=True,random_state=CONFIG['cv_random_state']); seen=np.zeros(len(y),int); subject_rows=[]
    for fold,(tr,te) in enumerate(splitter.split(np.zeros(len(y)),y)):
        pred,score,dist=fit_riemann_fold(views,y,tr,te); seen[te]+=1
        for j,k in enumerate(te): subject_rows.append({'subject_id':sid,'trial_index':int(k),'trial_id':d['trial_ids'][k],'fold_id':fold,'label':int(y[k]),'prediction':int(pred[j]),'score':float(score[j]),'distance_class0':float(dist[j,0]),'distance_class1':float(dist[j,1])})
    assert np.all(seen==1); RIEMANN_ROWS.extend(subject_rows); r=pd.DataFrame(subject_rows).sort_values('trial_index'); a=r[r.label==0].score.to_numpy(); b=r[r.label==1].score.to_numpy(); RIEMANN_SEP.append({'subject_id':sid,'balanced_accuracy':balanced_accuracy_score(r.label,r.prediction),'auc':roc_auc_score(r.label,r.score),'margin_hedges_g':hedges_g(a,b),'margin_rank_biserial':rank_biserial(a,b),'margin_centroid_distance':abs(b.mean()-a.mean()),'mean_foldlocal_distance_contrast':float(np.mean(np.where(r.label==1,r.distance_class0-r.distance_class1,r.distance_class1-r.distance_class0)))})
RIEMANN=pd.DataFrame(RIEMANN_ROWS); RIEMANN_SEP=pd.DataFrame(RIEMANN_SEP)
SJEPA=pd.DataFrame(); SJEPA_SEP=pd.DataFrame(); SJEPA_DIM=pd.DataFrame(); SJEPA_MANIFEST=None; SJEPA_BY_SUBJECT={}
if CONFIG['enable_sjepa']:
    emb,sy,sids,ssub,SJEPA_MANIFEST=extract_sjepa(ALL_DATA); rows=[]; dim=[]; sep=[]
    for sid in SUBJECTS:
        mask=ssub==sid; x=emb[mask]; y=sy[mask]; ids=sids[mask]; SJEPA_BY_SUBJECT[sid]=x; rr=pd.DataFrame(fit_vector_oof(x,y,sid,'sjepa')).sort_values('trial_index'); rr['trial_id']=ids[rr.trial_index.to_numpy()]; rows.append(rr); a=rr[rr.label==0].score.to_numpy(); b=rr[rr.label==1].score.to_numpy(); sep.append({'subject_id':sid,'balanced_accuracy':balanced_accuracy_score(rr.label,rr.prediction),'auc':roc_auc_score(rr.label,rr.score),'margin_hedges_g':hedges_g(a,b),'margin_rank_biserial':rank_biserial(a,b),'linear_mmd_sq':float(np.sum((x[y==1].mean(0)-x[y==0].mean(0))**2)),'centroid_distance':float(np.linalg.norm(x[y==1].mean(0)-x[y==0].mean(0)))})
        observed_mmd=float(np.sum((x[y==1].mean(0)-x[y==0].mean(0))**2)); prng=np.random.default_rng(BASE_SEED+sid); exceed=0
        for _ in range(CONFIG['permutation_iterations']):
            yp=prng.permutation(y); exceed += float(np.sum((x[yp==1].mean(0)-x[yp==0].mean(0))**2)) >= observed_mmd-1e-15
        sep[-1]['linear_mmd_permutation_p']=(exceed+1)/(CONFIG['permutation_iterations']+1)
        for j in range(x.shape[1]):
            aa=x[y==0,j]; bb=x[y==1,j]; dim.append({'subject_id':sid,'dimension':j,'hedges_g':hedges_g(aa,bb),'rank_biserial':rank_biserial(aa,bb),'difference_right_minus_left':bb.mean()-aa.mean(),'permutation_p':permutation_p(aa,bb,lambda q,w:np.mean(w)-np.mean(q),np.random.default_rng(BASE_SEED+sid*100+j))})
    SJEPA=pd.concat(rows,ignore_index=True); SJEPA_SEP=pd.DataFrame(sep); SJEPA_DIM=pd.DataFrame(dim); SJEPA_DIM['bh_q']=SJEPA_DIM.groupby('subject_id').permutation_p.transform(lambda p:bh(p)); SJEPA_DIM['correction']='BH across 64 dimensions within subject'
MICRO_LONG,MATCHING=build_microstates(ALL_DATA); MICRO_STATS=[]; rng=np.random.default_rng(CONFIG['bootstrap_seed'])
for feature,g in MICRO_LONG.groupby('feature'):
    pivot=g.pivot(index='subject_id',columns='label',values='value').loc[SUBJECTS]; diff=pivot[1]-pivot[0]; boots=[np.mean(rng.choice(diff,len(diff),True)) for _ in range(CONFIG['bootstrap_iterations'])]; t=stats.ttest_rel(pivot[1],pivot[0]); w=stats.wilcoxon(diff) if np.any(diff) else type('R',(),{'statistic':0,'pvalue':1})(); MICRO_STATS.append({'feature':feature,'mean_left':pivot[0].mean(),'mean_right':pivot[1].mean(),'paired_difference':diff.mean(),'cohen_dz':diff.mean()/(diff.std(ddof=1)+1e-12),'bootstrap_ci_low':np.quantile(boots,.025),'bootstrap_ci_high':np.quantile(boots,.975),'paired_t_stat':t.statistic,'paired_t_p':t.pvalue,'wilcoxon_stat':w.statistic,'wilcoxon_p':w.pvalue,'transductive_group_maps':True,'state_naming_mode':CONFIG['state_naming_mode']})
MICRO_STATS=pd.DataFrame(MICRO_STATS); MICRO_STATS['bh_q']=bh(MICRO_STATS.wilcoxon_p)
paper_features=['A_duration','A_occurrence','A_coverage','C_duration','C_occurrence','C_coverage','B_to_A','D_to_A','D_to_C']; expected={'A_duration':('left_greater',79.04,73.20,0.84,0.032,0.14),'A_occurrence':('left_greater',2.80,2.59,0.82,0.018,0.029),'A_coverage':('left_greater',22.85,19.75,1.16,0.004,0.06),'C_duration':('right_greater',82.60,88.93,-0.70,0.044,0.14),'C_occurrence':('right_greater',2.79,3.23,-1.14,0.004,0.53),'C_coverage':('right_greater',23.35,29.18,-1.03,0.004,0.33),'B_to_A':('left_greater',32.15,29.09,0.82,0.040,0.025),'D_to_A':('left_greater',34.24,27.79,1.25,0.001,0.26),'D_to_C':('right_greater',34.28,43.85,-1.22,0.006,0.20)}
if CONFIG['state_naming_mode']=='proxy_canonical':
    PAPER=MICRO_STATS[MICRO_STATS.feature.isin(paper_features)].copy(); PAPER['lv_expected_direction']=PAPER.feature.map(lambda f:expected[f][0]); PAPER['lv_mean_left']=PAPER.feature.map(lambda f:expected[f][1]); PAPER['lv_mean_right']=PAPER.feature.map(lambda f:expected[f][2]); PAPER['lv_cohens_d_left_minus_right']=PAPER.feature.map(lambda f:expected[f][3]); PAPER['lv_fdr_p']=PAPER.feature.map(lambda f:expected[f][4]); PAPER['lv_fisher_score']=PAPER.feature.map(lambda f:expected[f][5]); PAPER['direction_agreement']=PAPER.apply(lambda r:(r.paired_difference<0)==(r.lv_expected_direction=='left_greater'),axis=1); PAPER['comparison_status']='proxy_canonical_not_exact'
else: PAPER=pd.DataFrame([{'feature':f,'lv_expected_direction':None,'lv_mean_left':np.nan,'lv_mean_right':np.nan,'lv_cohens_d_left_minus_right':np.nan,'lv_fdr_p':np.nan,'lv_fisher_score':np.nan,'direction_agreement':np.nan,'comparison_status':'unavailable_neutral_states_no_canonical_claim'} for f in paper_features])

# 6. Results
## 6.1 Cross-Branch Summary, Correlations, and Split-Half Generalization

In [ ]:
def locked_performance():
    out=pd.DataFrame({'subject_id':SUBJECTS}); shallow=Path(CONFIG['locked_shallow_subject_results']); pre=Path(CONFIG['locked_prelocal_subject_metrics']); availability={}
    if shallow.exists():
        z=pd.read_csv(shallow); z['subject_id']=z.subject_id.astype(str).str.extract(r'(\d+)')[0].astype(int); z=z[z.subject_id.isin(SUBJECTS)]; out=out.merge(z[['subject_id','balanced_accuracy']].drop_duplicates('subject_id').rename(columns={'balanced_accuracy':'locked_shallow_ba'}),on='subject_id',how='left'); folds=shallow.with_name('fold_results.csv')
        if folds.exists():
            import ast
            f=pd.read_csv(folds); f['subject_id']=f.subject_id.astype(str).str.extract(r'(\d+)')[0].astype(int); f=f[f.subject_id.isin(SUBJECTS)]; f['collapsed']=f.collapse_diagnostics.map(lambda v:bool(ast.literal_eval(v).get('collapsed',False))); out=out.merge(f.groupby('subject_id').collapsed.mean().rename('locked_shallow_collapse_rate'),on='subject_id',how='left')
        else: out['locked_shallow_collapse_rate']=np.nan
    else: out['locked_shallow_ba']=np.nan; out['locked_shallow_collapse_rate']=np.nan
    availability['locked_shallow']={'path':str(shallow),'file_exists':shallow.exists(),'available_subjects':out.loc[out.locked_shallow_ba.notna(),'subject_id'].astype(int).tolist(),'missing_subjects':out.loc[out.locked_shallow_ba.isna(),'subject_id'].astype(int).tolist()}
    if pre.exists():
        q=json.loads(pre.read_text()); values={int(str(k).replace('sub-','')):v.get('mean_balanced_accuracy') for k,v in q.items() if str(k).replace('sub-','').isdigit()}; out['locked_prelocal_ba']=out.subject_id.map(values)
    else: out['locked_prelocal_ba']=np.nan
    availability['locked_prelocal']={'path':str(pre),'file_exists':pre.exists(),'available_subjects':out.loc[out.locked_prelocal_ba.notna(),'subject_id'].astype(int).tolist(),'missing_subjects':out.loc[out.locked_prelocal_ba.isna(),'subject_id'].astype(int).tolist()}
    return out,availability
phys_mag=PHYS_EFFECTS.groupby('subject_id').hedges_g.apply(lambda x:np.mean(np.abs(x))).rename('physiology_mean_abs_g'); micro_mag=MICRO_LONG.pivot_table(index=['subject_id','feature'],columns='label',values='value').assign(d=lambda x:(x[1]-x[0]).abs()).groupby('subject_id').d.mean().rename('microstate_mean_abs_difference')
SUMMARY,EXTERNAL_METRICS_AVAILABILITY=locked_performance(); SUMMARY=SUMMARY.merge(phys_mag,on='subject_id').merge(RIEMANN_SEP.rename(columns={'balanced_accuracy':'riemann_ba','auc':'riemann_auc','margin_centroid_distance':'riemann_separation'}),on='subject_id').merge(micro_mag,on='subject_id')
if CONFIG['enable_sjepa']: SUMMARY=SUMMARY.merge(SJEPA_SEP.rename(columns={'balanced_accuracy':'sjepa_ba','auc':'sjepa_auc','centroid_distance':'sjepa_separation'}),on='subject_id')
SUMMARY['flagged_sub15_sensitivity']=SUMMARY.subject_id.eq(15)
CORR_SPECS=[('physiology_mean_abs_g','locked_shallow_ba'),('riemann_separation','riemann_ba'),('microstate_mean_abs_difference','locked_shallow_ba')]+([('sjepa_separation','sjepa_ba')] if CONFIG['enable_sjepa'] else [])
CORRS=[]; rng=np.random.default_rng(CONFIG['bootstrap_seed'])
for i,(xcol,ycol) in enumerate(CORR_SPECS):
    z=SUMMARY[['subject_id',xcol,ycol]].dropna(); x=z[xcol].to_numpy(); y=z[ycol].to_numpy()
    for method in ['spearman','kendall']:
        est,p,permutation_method=permutation_corr(x,y,method,CONFIG['correlation_permutations'],BASE_SEED+i); fn=stats.spearmanr if method=='spearman' else stats.kendalltau; boots=[]
        for _ in range(CONFIG['bootstrap_iterations']):
            ii=rng.integers(0,len(x),len(x)); val=float(fn(x[ii],y[ii]).statistic); boots.append(val if np.isfinite(val) else 0.)
        loo=[float(fn(np.delete(x,j),np.delete(y,j)).statistic) for j in range(len(x))]; CORRS.append({'separation_metric':xcol,'performance_metric':ycol,'method':method,'estimate':est,'exact_permutation_p':p,'permutation_method':permutation_method,'bootstrap_ci_low':np.quantile(boots,.025),'bootstrap_ci_high':np.quantile(boots,.975),'loo_min':np.min(loo),'loo_max':np.max(loo),'loo_same_sign':bool(all(np.sign(v)==np.sign(est) for v in loo)),'same_dataset_association':True,'n_subjects':len(x)})
CORRS=pd.DataFrame(CORRS); order=np.argsort(CORRS.exact_permutation_p); adjusted=np.empty(len(CORRS)); adjusted[order]=np.maximum.accumulate(np.minimum(1,CORRS.exact_permutation_p.to_numpy()[order]*(len(CORRS)-np.arange(len(CORRS))))); CORRS['holm_p']=adjusted
def split_half_vector(X,y,rng):
    train=np.r_[rng.choice(np.flatnonzero(y==0),len(np.flatnonzero(y==0))//2,False),rng.choice(np.flatnonzero(y==1),len(np.flatnonzero(y==1))//2,False)]; test=np.setdiff1d(np.arange(len(y)),train); scaler=StandardScaler().fit(X[train]); a,b=scaler.transform(X[train]),scaler.transform(X[test]); sep=np.linalg.norm(a[y[train]==1].mean(0)-a[y[train]==0].mean(0)); clf=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto').fit(a,y[train]); return train,test,sep,balanced_accuracy_score(y[test],clf.predict(b))
def split_half_riemann(views,y,train,test):
    pred,score,_=fit_riemann_fold(views,y,train,test); from pyriemann.tangentspace import TangentSpace; parts=[]
    for name in ['4s','2s','1s']: parts.append(TangentSpace(metric='riemann').fit_transform(views[name][train]))
    a=np.c_[*parts]; a=StandardScaler().fit_transform(a); sep=np.linalg.norm(a[y[train]==1].mean(0)-a[y[train]==0].mean(0)); return sep,balanced_accuracy_score(y[test],pred)
SPLIT=[]
for repeat in range(CONFIG['split_half_repeats']):
    seps={'physiology':[],'sjepa':[]}; bas={'physiology':[],'sjepa':[]}; seed=CONFIG['split_half_seed']+repeat
    for d in ALL_DATA:
        sid=d['subject_id']; y=d['labels']; p=PHYS[(PHYS.subject_id==sid)&(PHYS.band=='8-30Hz')].sort_values('trial_id'); cols=[c for c in p if c.startswith(('logbp_','asym_','motor13_'))]; Xp=p[cols].to_numpy(); rngs=np.random.default_rng(seed+sid); tr,te,sep,ba=split_half_vector(Xp,y,rngs); seps['physiology'].append(sep); bas['physiology'].append(ba); SPLIT.append({'repeat':repeat,'swap':0,'subject_id':sid,'branch':'physiology','train_separation':sep,'heldout_ba':ba,'train_indices':json.dumps(tr.tolist()),'test_indices':json.dumps(te.tolist())}); scaler=StandardScaler().fit(Xp[te]); aa,bb=scaler.transform(Xp[te]),scaler.transform(Xp[tr]); sep2=np.linalg.norm(aa[y[te]==1].mean(0)-aa[y[te]==0].mean(0)); clf=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto').fit(aa,y[te]); ba2=balanced_accuracy_score(y[tr],clf.predict(bb)); SPLIT.append({'repeat':repeat,'swap':1,'subject_id':sid,'branch':'physiology','train_separation':sep2,'heldout_ba':ba2,'train_indices':json.dumps(te.tolist()),'test_indices':json.dumps(tr.tolist())}); rsep,rba=split_half_riemann(RIEMANN_VIEWS[sid],y,tr,te); SPLIT.append({'repeat':repeat,'swap':0,'subject_id':sid,'branch':'riemann','train_separation':rsep,'heldout_ba':rba,'train_indices':json.dumps(tr.tolist()),'test_indices':json.dumps(te.tolist())}); rsep,rba=split_half_riemann(RIEMANN_VIEWS[sid],y,te,tr); SPLIT.append({'repeat':repeat,'swap':1,'subject_id':sid,'branch':'riemann','train_separation':rsep,'heldout_ba':rba,'train_indices':json.dumps(te.tolist()),'test_indices':json.dumps(tr.tolist())})
        if CONFIG['enable_sjepa']:
            xs=SJEPA_BY_SUBJECT[sid]; tr,te,sep,ba=split_half_vector(xs,y,np.random.default_rng(seed+1000+sid)); SPLIT.append({'repeat':repeat,'swap':0,'subject_id':sid,'branch':'sjepa','train_separation':sep,'heldout_ba':ba,'train_indices':json.dumps(tr.tolist()),'test_indices':json.dumps(te.tolist())}); scaler=StandardScaler().fit(xs[te]); aa,bb=scaler.transform(xs[te]),scaler.transform(xs[tr]); sep2=np.linalg.norm(aa[y[te]==1].mean(0)-aa[y[te]==0].mean(0)); clf=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto').fit(aa,y[te]); ba2=balanced_accuracy_score(y[tr],clf.predict(bb)); SPLIT.append({'repeat':repeat,'swap':1,'subject_id':sid,'branch':'sjepa','train_separation':sep2,'heldout_ba':ba2,'train_indices':json.dumps(te.tolist()),'test_indices':json.dumps(tr.tolist())})
SPLIT=pd.DataFrame(SPLIT); split_summary=[]
for (branch,repeat,swap),g in SPLIT.groupby(['branch','repeat','swap']): split_summary.append({'branch':branch,'repeat':repeat,'swap':swap,'spearman':stats.spearmanr(g.train_separation,g.heldout_ba).statistic,'kendall':stats.kendalltau(g.train_separation,g.heldout_ba).statistic,'mean_heldout_ba':g.heldout_ba.mean()})
SPLIT_SUMMARY=pd.DataFrame(split_summary)

## 6.2 Required Visualizations

In [ ]:
PLOTS={}; TOPOMAP_PAGES={}; TOPOMAP_ROWS=[]
heat=PHYS_EFFECTS[PHYS_EFFECTS.band=='8-30Hz'].pivot_table(index='subject_id',columns='feature',values='hedges_g').loc[SUBJECTS]; side=SUMMARY.set_index('subject_id').loc[SUBJECTS,[c for c in ['locked_shallow_ba','locked_prelocal_ba','riemann_ba','riemann_auc','sjepa_ba','sjepa_auc','locked_shallow_collapse_rate'] if c in SUMMARY]].copy(); fig,(ax,sax)=plt.subplots(1,2,figsize=(22,max(7,len(SUBJECTS)*.25)),gridspec_kw={'width_ratios':[8,2]}); sns.heatmap(heat,center=0,cmap='vlag',ax=ax); sns.heatmap(side,cmap='viridis',vmin=0,vmax=1,annot=len(SUBJECTS)<=20,fmt='.2f',ax=sax,cbar=False); ax.set_title('Signed Hedges g: right-hand minus left-hand MI-window bandpower'); sax.set_title('Available BA / AUC / collapse metrics'); ax.set_ylabel('subject ID'); fig.tight_layout(); PLOTS['subject_feature_heatmap']=str(ARTIFACT_DIR/'subject_feature_signed_effect_heatmap.png'); fig.savefig(PLOTS['subject_feature_heatmap'],dpi=180); plt.close(fig)
fig,axes=plt.subplots(2,2,figsize=(12,10)); sub=PHYS[PHYS.band=='8-30Hz'];
for ax,(a,b) in zip(axes.flat,PAIRS):
    q=sub.groupby(['subject_id','label'])[[f'logbp_{a}',f'logbp_{b}']].mean().reset_index(); sns.scatterplot(data=q,x=f'logbp_{a}',y=f'logbp_{b}',hue='label',ax=ax); ax.set_title(f'{a}/{b}: MI-window log bandpower')
fig.tight_layout(); PLOTS['physiology_panels']=str(ARTIFACT_DIR/'physiology_homologous_pair_panels.png'); fig.savefig(PLOTS['physiology_panels'],dpi=180); plt.close(fig)
plot_features=paper_features if CONFIG['state_naming_mode']=='proxy_canonical' else sorted(MICRO_LONG.feature.unique())[:9]; fig,axes=plt.subplots(3,3,figsize=(13,11));
for ax,f in zip(axes.flat,plot_features):
    p=MICRO_LONG[MICRO_LONG.feature==f].pivot(index='subject_id',columns='label',values='value').loc[SUBJECTS]; [ax.plot([0,1],p.loc[s],marker='o',alpha=.45) for s in SUBJECTS]; ax.set_xticks([0,1],['left','right']); ax.set_title(f)
fig.suptitle('Microstate paired differences: proxy only' if CONFIG['state_naming_mode']=='proxy_canonical' else 'Neutral-state equivalents; no canonical A/B/C/D interpretation'); fig.tight_layout(); PLOTS['microstate_paired']=str(ARTIFACT_DIR/'microstate_nine_paired_plots.png'); fig.savefig(PLOTS['microstate_paired'],dpi=180); plt.close(fig)
fig,axes=plt.subplots(2,1,figsize=(max(15,len(SUBJECTS)*.35),9)); sns.violinplot(data=RIEMANN,x='subject_id',y='score',hue='label',split=True,ax=axes[0]); axes[0].set_title('Held-out Riemann OOF score distributions');
if CONFIG['enable_sjepa']: sns.violinplot(data=SJEPA,x='subject_id',y='score',hue='label',split=True,ax=axes[1]); axes[1].set_title('Held-out frozen S-JEPA OOF score distributions')
fig.tight_layout(); PLOTS['oof_scores']=str(ARTIFACT_DIR/'heldout_score_distributions.png'); fig.savefig(PLOTS['oof_scores'],dpi=180); plt.close(fig)
if CONFIG['enable_sjepa']:
    z=PCA(2).fit_transform(emb); fig,ax=plt.subplots(figsize=(9,7)); ax.scatter(z[:,0],z[:,1],c=sy,cmap='coolwarm',alpha=.55); ax.set_title('Descriptive all-cohort transductive PCA of frozen S-JEPA embeddings; not inference'); fig.tight_layout(); PLOTS['sjepa_pca']=str(ARTIFACT_DIR/'sjepa_descriptive_transductive_pca.png'); fig.savefig(PLOTS['sjepa_pca'],dpi=180); plt.close(fig)
fig,axes=plt.subplots(1,len(CORR_SPECS),figsize=(5*len(CORR_SPECS),4)); axes=np.atleast_1d(axes)
for ax,(x,y) in zip(axes,CORR_SPECS): sns.regplot(data=SUMMARY,x=x,y=y,ax=ax,ci=None); [ax.text(r[x],r[y],str(int(r.subject_id)),fontsize=8) for _,r in SUMMARY.dropna(subset=[x,y]).iterrows()]; c=CORRS[(CORRS.separation_metric==x)&(CORRS.performance_metric==y)&(CORRS.method=='spearman')].iloc[0]; ax.set_title(f'rho={c.estimate:.2f}; LOO [{c.loo_min:.2f},{c.loo_max:.2f}]')
fig.tight_layout(); PLOTS['correlations']=str(ARTIFACT_DIR/'separability_vs_performance_loo.png'); fig.savefig(PLOTS['correlations'],dpi=180); plt.close(fig)
spaces=SUMMARY.select_dtypes('number').drop(columns=['subject_id'],errors='ignore'); fig,ax=plt.subplots(figsize=(12,9)); sns.heatmap(spaces.corr(method='spearman'),center=0,cmap='vlag',annot=True,fmt='.2f',ax=ax); ax.set_title(f'Cross-space concordance ({CONFIG["cohort_name"]} configured cohort)'); fig.tight_layout(); PLOTS['concordance']=str(ARTIFACT_DIR/'cross_space_concordance_heatmap.png'); fig.savefig(PLOTS['concordance'],dpi=180); plt.close(fig)

import mne
TOPOMAP_INFO=mne.create_info(LIU29,CONFIG['sfreq'],ch_types='eeg'); TOPOMAP_INFO.set_montage(mne.channels.make_standard_montage('standard_1020'),match_case=False,on_missing='raise')
def robust_limit(values):
    finite=np.abs(np.asarray(values,float)); finite=finite[np.isfinite(finite)]; return max(float(np.quantile(finite,.98)) if len(finite) else 1.,1e-9)
def draw_topomap(ax,values,vlim,title):
    mne.viz.plot_topomap(np.asarray(values,float),TOPOMAP_INFO,axes=ax,show=False,cmap='RdBu_r',vlim=(-vlim,vlim),contours=0,sensors=True); ax.set_title(title,fontsize=9)
bands=[f'{lo}-{hi}Hz' for lo,hi in CONFIG['bands_hz']]
cohort_g=[]; cohort_d=[]
for band in bands:
    q=PHYS_CHANNEL_EFFECTS[PHYS_CHANNEL_EFFECTS.band==band].pivot(index='subject_id',columns='channel',values='hedges_g').loc[SUBJECTS,LIU29]; cohort_g.append(q.mean().to_numpy());
    d=PHYS_CHANNEL_EFFECTS[PHYS_CHANNEL_EFFECTS.band==band].pivot(index='subject_id',columns='channel',values='difference_right_minus_left').loc[SUBJECTS,LIU29]; cohort_d.append(d.mean().to_numpy())
for values,filename,key,metric in [(cohort_g,'cohort_bandpower_effect_topomaps.png','cohort_bandpower_effect_topomaps','cohort_mean_signed_hedges_g'),(cohort_d,'cohort_bandpower_difference_topomaps.png','cohort_bandpower_difference_topomaps','cohort_mean_log_bandpower_difference')]:
    lim=robust_limit(np.concatenate(values)); fig,axes=plt.subplots(1,len(bands),figsize=(4*len(bands),3.8));
    for ax,band,v in zip(np.atleast_1d(axes),bands,values):
        draw_topomap(ax,v,lim,f'{band}\nright minus left ({metric})')
        for ch,value in zip(LIU29,v): TOPOMAP_ROWS.append({'plot':filename,'page':1,'selection_reason':'configured_cohort_mean','subject_id':np.nan,'band':band,'channel':ch,'metric':metric,'value':value,'polarity':'right-hand minus left-hand MI-window log bandpower'})
    fig.suptitle('MI-window bandpower: right-hand minus left-hand; positive is higher for right-hand trials'); fig.tight_layout(); PLOTS[key]=str(ARTIFACT_DIR/filename); fig.savefig(PLOTS[key],dpi=180); plt.close(fig)
for band in bands:
    q=PHYS_CHANNEL_EFFECTS[PHYS_CHANNEL_EFFECTS.band==band].pivot(index='subject_id',columns='channel',values='hedges_g').loc[SUBJECTS,LIU29]; lim=robust_limit(q.to_numpy()); pages=[]
    for page,start in enumerate(range(0,len(SUBJECTS),25),1):
        page_subjects=SUBJECTS[start:start+25]; fig,axes=plt.subplots(5,5,figsize=(14,15));
        for ax in axes.flat: ax.axis('off')
        for ax,sid in zip(axes.flat,page_subjects): ax.axis('on'); draw_topomap(ax,q.loc[sid].to_numpy(),lim,f'sub-{sid:02d}')
        safe=band.replace('-','_').replace('Hz','hz'); filename=f'subject_bandpower_effect_topomaps_{safe}.png' if page==1 else f'subject_bandpower_effect_topomaps_{safe}_page{page:02d}.png'; fig.suptitle(f'{band} signed Hedges g, right-hand minus left-hand MI-window bandpower; common vlim +/-{lim:.2f}'); fig.tight_layout(); path=ARTIFACT_DIR/filename; fig.savefig(path,dpi=160); plt.close(fig); pages.append(str(path))
        for sid in page_subjects:
            for ch,value in q.loc[sid].items(): TOPOMAP_ROWS.append({'plot':filename,'page':page,'selection_reason':'all_configured_subjects','subject_id':sid,'band':band,'channel':ch,'metric':'subject_signed_hedges_g','value':value,'polarity':'right-hand minus left-hand MI-window log bandpower'})
    TOPOMAP_PAGES[band]=pages
representatives=[sid for sid in [1,7,11,14,29,31,37,41] if sid in SUBJECTS]; selected_band='8-30Hz'; q=PHYS_CHANNEL_EFFECTS[PHYS_CHANNEL_EFFECTS.band==selected_band].pivot(index='subject_id',columns='channel',values='hedges_g').loc[representatives,LIU29]; lim=robust_limit(q.to_numpy()); ncols=4; nrows=max(1,int(np.ceil(len(representatives)/ncols))); fig,axes=plt.subplots(nrows,ncols,figsize=(4*ncols,3.7*nrows)); axes=np.atleast_1d(axes).ravel()
for ax in axes: ax.axis('off')
for ax,sid in zip(axes,representatives): ax.axis('on'); draw_topomap(ax,q.loc[sid],lim,f'sub-{sid:02d} (fixed prespecified ID)')
fig.suptitle('Selected fixed subjects, 8-30 Hz signed Hedges g: right-hand minus left-hand MI-window bandpower'); fig.tight_layout(); PLOTS['subject_motor_topomaps_selected']=str(ARTIFACT_DIR/'subject_motor_topomaps_selected.png'); fig.savefig(PLOTS['subject_motor_topomaps_selected'],dpi=180); plt.close(fig)
for sid in representatives:
    for ch,value in q.loc[sid].items(): TOPOMAP_ROWS.append({'plot':'subject_motor_topomaps_selected.png','page':1,'selection_reason':'fixed_prespecified_ids_[1,7,11,14,29,31,37,41]','subject_id':sid,'band':selected_band,'channel':ch,'metric':'subject_signed_hedges_g','value':value,'polarity':'right-hand minus left-hand MI-window log bandpower'})
TOPOMAP_SUMMARY=pd.DataFrame(TOPOMAP_ROWS)


## 6.3 Save Artifacts and Experiment Summary

In [ ]:
TABLES={'physiology_trial_features.csv':PHYS,'physiology_subject_effects.csv':PHYS_EFFECTS,'physiology_channel_effects.csv':PHYS_CHANNEL_EFFECTS,'physiology_cohort_effects.csv':PHYS_COHORT_EFFECTS,'topomap_summary.csv':TOPOMAP_SUMMARY,'riemann_oof_predictions.csv':RIEMANN,'riemann_subject_separability.csv':RIEMANN_SEP,'sjepa_dimension_effects.csv':SJEPA_DIM,'sjepa_oof_predictions.csv':SJEPA,'sjepa_subject_separability.csv':SJEPA_SEP,'microstate_features_long.csv':MICRO_LONG,'microstate_class_differences.csv':MICRO_STATS,'microstate_paper_comparison.csv':PAPER,'state_matching_matrix.csv':MATCHING,'subject_separability_summary.csv':SUMMARY,'separability_performance_correlations.csv':CORRS,'split_half_generalization.csv':SPLIT}
for name,df in TABLES.items(): df.to_csv(ARTIFACT_DIR/name,index=False)
cohort_sig=PHYS_COHORT_EFFECTS.bh_q.lt(.05)
GLOBAL_METRICS={'primary_estimand':'within-subject left-vs-right separation and same-dataset association with decoding','cohort_name':CONFIG['cohort_name'],'cohort_label':'Full50' if SUBJECTS==list(range(1,51)) else CONFIG['cohort_name'],'subjects':SUBJECTS,'n_subjects':len(SUBJECTS),'flagged_subjects_present':sorted(set(SUBJECTS)&set(CONFIG['flagged_subjects'])),'physiology_always_run':True,'physiology_cohort_effects':{'n_tests':len(PHYS_COHORT_EFFECTS),'n_bh_significant':int(cohort_sig.sum()),'correction':'BH across all channel x band and prespecified homologous/asymmetry cohort tests'},'external_metrics_availability':EXTERNAL_METRICS_AVAILABILITY,'topomaps':{'cohort_effect':PLOTS['cohort_bandpower_effect_topomaps'],'cohort_difference':PLOTS['cohort_bandpower_difference_topomaps'],'subject_pages':TOPOMAP_PAGES,'selected_fixed_subjects':PLOTS['subject_motor_topomaps_selected'],'summary':str(ARTIFACT_DIR/'topomap_summary.csv')},'riemann':{'mean_ba':float(RIEMANN_SEP.balanced_accuracy.mean()),'mean_auc':float(RIEMANN_SEP.auc.mean()),'exactly_once_oof':bool(RIEMANN.groupby(['subject_id','trial_index']).size().eq(1).all())},'sjepa':{'enabled':CONFIG['enable_sjepa'],'available':not SJEPA.empty,'mean_ba':None if SJEPA_SEP.empty else float(SJEPA_SEP.balanced_accuracy.mean()),'mean_auc':None if SJEPA_SEP.empty else float(SJEPA_SEP.auc.mean()),'non_logit_embedding_dim':None if SJEPA.empty else 64,'exactly_once_oof':None if SJEPA.empty else bool(SJEPA.groupby(['subject_id','trial_index']).size().eq(1).all())},'microstate':{'n_features':int(MICRO_LONG.feature.nunique()),'off_diagonal_transitions':12,'self_transitions':0,'state_naming_mode':CONFIG['state_naming_mode'],'all_configured_cohort_transductive_group_maps':True,'inductive_classifier_evidence':False,'canonical_state_claim':False,'paper_transitions':['B_to_A','D_to_A','D_to_C']},'multiplicity':{'physiology_local':'BH within subject x band x declared family','physiology_cohort':'BH across all channel x band and prespecified homologous/asymmetry cohort tests','sjepa_dimensions':'BH within subject','microstate':'BH across 24 paired features','correlations':'Holm across prespecified method/specification family'},'same_dataset_associations':True,'split_half':{'repeats':CONFIG['split_half_repeats'],'swaps_per_repeat':2,'summary':SPLIT_SUMMARY.to_dict('records')}}
def json_safe(value):
    if isinstance(value,dict): return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)): return [json_safe(v) for v in value]
    if isinstance(value,np.integer): return int(value)
    if isinstance(value,(np.floating,float)): return None if not np.isfinite(value) else float(value)
    if isinstance(value,np.ndarray): return json_safe(value.tolist())
    return value
GLOBAL_METRICS=json_safe(GLOBAL_METRICS)
with open(ARTIFACT_DIR/'global_metrics.json','w') as f: json.dump(GLOBAL_METRICS,f,indent=2,allow_nan=False)
artifact_paths={name:str(ARTIFACT_DIR/name) for name in TABLES}; artifact_paths['sjepa_embeddings.npz']=str(ARTIFACT_DIR/'sjepa_embeddings.npz') if CONFIG['enable_sjepa'] else None; artifact_paths.update(PLOTS); artifact_paths['subject_bandpower_effect_topomap_pages']=TOPOMAP_PAGES; artifact_paths.update({'config':str(config_path),'subject_inventory':str(ARTIFACT_DIR/'subject_inventory.csv'),'global_metrics':str(ARTIFACT_DIR/'global_metrics.json'),'run_metadata':str(ARTIFACT_DIR/'run_metadata.json'),'run_log':str(LOG_PATH)})
RUN_METADATA={'run_id':RUN_ID,'artifact_dir':str(ARTIFACT_DIR),'experiment_name':CONFIG['experiment_name'],'config_note':CONFIG['config_note'],'cohort_name':CONFIG['cohort_name'],'subjects':SUBJECTS,'flagged_subjects':CONFIG['flagged_subjects'],'n_channels':29,'channel_names':LIU29,'preprocessing_order':'independent full 8-s trial filter, then marker-relative 0-4 s crop','topomap_interpretation':'MI-window log bandpower right-hand minus left-hand; positive values are higher for right-hand trials and are not baseline-normalized ERD','external_metrics_availability':EXTERNAL_METRICS_AVAILABILITY,'model':{'riemann':'fixed motor13 OAS 8-30 Hz 4s+2s+1s fold-local tangent/scaler/shrinkage-LDA','sjepa':SJEPA_MANIFEST,'microstate':'four-state all-configured-cohort transductive group maps; neutral states make no canonical claim'},'seeds':{'base':BASE_SEED,'cv':CONFIG['cv_random_state'],'bootstrap':CONFIG['bootstrap_seed'],'split_half':CONFIG['split_half_seed']},'artifacts':artifact_paths,'global_metrics':GLOBAL_METRICS}
RUN_METADATA=json_safe(RUN_METADATA)
with open(ARTIFACT_DIR/'run_metadata.json','w') as f: json.dump(RUN_METADATA,f,indent=2,allow_nan=False)
print(f"Completed {CONFIG['cohort_name']} audit ({len(SUBJECTS)} subjects): Riemann BA={GLOBAL_METRICS['riemann']['mean_ba']:.3f}; S-JEPA available={GLOBAL_METRICS['sjepa']['available']}; microstates are all-cohort transductive.")
print(f'Global metrics saved to:  {ARTIFACT_DIR / "global_metrics.json"}')
print(f'Run metadata saved to:    {ARTIFACT_DIR / "run_metadata.json"}')
print(f'\nAll artifacts in: {ARTIFACT_DIR}')
try: _LOG_FILE_HANDLE.close()
except Exception: pass
